# Scoring
> Score a sampling campaign against the ground truth

Work in progress, following `scoring-plan.md` (#12): campaigns, strategies and scores for comparing sampling strategies. The comparisons themselves, which take minutes, are in `_scoring_experiments.ipynb`.


In [ ]:
import math, os, tempfile
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, beta, binom
import geopandas as gpd
import shapely
import rasterio
from rasterio.enums import Resampling
from fastcore.test import *
from fastcore.utils import store_attr, patch

from trufl.utils import gridder, zonal_values
from trufl.sampler import Sampler, rank_to_sample
from trufl.collector import DataCollector
from trufl.callbacks import State, MaxCB, MinCB, StdCB, CountCB, MoranICB, PriorCB
from trufl.optimizer import Optimizer

One campaign to experiment on: a 10 x 10 grid over the simulated raster, with between 0 and 9 samples per unit, so some units have none:

In [ ]:
fname_raster = 'files/ground-truth-01-4326-simulated.tif'
gdf_grid = gridder(fname_raster, nrows=10, ncols=10)
n = np.random.default_rng(0).integers(0, 10, size=len(gdf_grid))
samples = DataCollector(fname_raster).collect(Sampler(gdf_grid).sample(n, method='uniform', rng=0))
samples.head()

`unit_values` reads the cells of a raster inside every unit once, as a Series of arrays indexed by `loc_id`. `Campaign` reads the ground truth with it, and the score functions below take the result in place of the raster:

In [ ]:
def unit_values(
    fname_raster:str, # Raster to read, such as the ground truth or a prior
    gdf_units:gpd.GeoDataFrame, # Units indexed by `loc_id`
) -> pd.Series: # Raster cell values of each unit, indexed by `loc_id`
    "Raster cell values inside each unit of `gdf_units`."
    return pd.Series({loc_id: zonal_values(fname_raster, geom) for loc_id, geom in gdf_units.geometry.items()}).rename_axis('loc_id')

`unit_stats` compares each unit's samples with its true values. Units without samples have a count of 0 and `NaN` for the measured values and the distance:

In [ ]:
def unit_stats(
    samples:gpd.GeoDataFrame, # Measurements indexed by `loc_id`, with a `value` column
    truth:pd.Series, # Raster cell values of each unit, from `unit_values`
) -> pd.DataFrame: # One row per unit
    "True and measured mean and maximum, sample count and Wasserstein distance of each unit."
    rows = {}
    for loc_id, true in truth.items():
        meas = samples.value[samples.index == loc_id].values
        rows[loc_id] = dict(true_mean=true.mean(), true_max=true.max(),
                            meas_mean=meas.mean() if len(meas) else np.nan, meas_max=meas.max() if len(meas) else np.nan,
                            n=len(meas), wasserstein=wasserstein_distance(true, meas) if len(meas) else np.nan)
    return pd.DataFrame.from_dict(rows, orient='index').rename_axis('loc_id')

In [ ]:
df = unit_stats(samples, unit_values(fname_raster, gdf_grid))
test_eq(df.n.sum(), len(samples))
df.head()

## Campaigns

A `Campaign` simulates a campaign over a ground-truth raster, one round at a time. A round can be a day, a week or a year. `Campaign` draws and measures samples but chooses no allocation: a strategy chooses it. It reads the true values once, and `score` turns the samples so far and the true values into one row per unit, `unit_stats` by default. `placement` chooses where samples go inside a unit; by default they are uniform random, and "Placement within a unit" below gives an alternative.

In [ ]:
class Campaign:
    "Simulated campaign over `fname_raster`, sampling the units of `gdf_units` one round at a time."
    def __init__(self,
        fname_raster:str, # Ground-truth raster
        gdf_units:gpd.GeoDataFrame, # Units indexed by `loc_id`
        score:callable=unit_stats, # Takes the samples so far and `unit_values`; returns one row per unit
        placement:callable=None, # Takes a unit's geometry, its earlier sample coordinates, a count and a random generator; returns coordinates. `None` places samples uniformly at random
    ):
        store_attr()
        self.sampler, self.collector = Sampler(gdf_units), DataCollector(fname_raster)
        self.truth = unit_values(fname_raster, gdf_units)

`reset` starts a campaign with no samples. It also seeds NumPy's global random state, which `MoranICB` uses for its permutation test because `esda.Moran` takes no seed:

In [ ]:
@patch
def reset(self:Campaign,
    seed:int=None, # Seed for sampling, strategies and `MoranICB`'s permutation test
) -> gpd.GeoDataFrame: # The samples so far: none
    "Start a new campaign with no samples."
    self.rng, self.rounds = np.random.default_rng(seed), []
    np.random.seed(seed)
    return self.samples

`samples` holds the measurements of every round so far, with the round each one was taken in:

In [ ]:
@patch(as_prop=True)
def samples(self:Campaign) -> gpd.GeoDataFrame: # Measurements indexed by `loc_id`, with `value` and `round` columns
    if not self.rounds: return gpd.GeoDataFrame(index=pd.Index([], name='loc_id'), geometry=None, data={'value': None, 'round': None})
    return pd.concat(self.rounds)

`step` draws and measures the samples of one round, then scores all samples so far with `score`:

In [ ]:
@patch
def step(self:Campaign,
    n:np.ndarray, # Number of samples for each unit, in `gdf_units` order
) -> pd.DataFrame: # `score` of all samples so far
    "Draw and measure `n` samples per unit, then score all samples so far."
    pts = self.sampler.sample(n, method='uniform', rng=self.rng) if self.placement is None else self.place(n)
    self.rounds.append(self.collector.collect(pts).assign(round=len(self.rounds) + 1))
    return self.score(self.samples, self.truth)

### Placement within a unit

With a spatially correlated ground truth, samples spread over a unit tell more than uniform random ones (see `_explain_06_spatial_sampling.ipynb`). Strata built anew each round cannot spread them, because a strategy usually gives a unit only one or two samples per round. `spread` places each new sample away from all the unit's earlier samples instead. It uses best-candidate sampling: draw `m` uniform candidates in the unit and keep the one farthest from the samples already there. With `m=1` it is uniform placement.

For a plume, distances can be measured in stretched coordinates: along direction `angle` (degrees counterclockwise from east) divided by `aspect`, the ratio of the range along the plume to the range across it. Samples then spread more across the plume than along it.

In [ ]:
def spread(
    m:int=10, # Candidates per sample; 1 gives uniform placement
    angle:float=0, # Plume direction, degrees counterclockwise from east
    aspect:float=1., # Range along the plume divided by the range across it
) -> callable: # A `placement` for `Campaign`
    "Best-candidate placement: each new sample is the farthest of `m` uniform candidates from the unit's earlier samples."
    t = np.deg2rad(angle)
    def stretch(xy): return np.c_[(xy[:, 0] * np.cos(t) + xy[:, 1] * np.sin(t)) / aspect, -xy[:, 0] * np.sin(t) + xy[:, 1] * np.cos(t)]
    def place(geom, prev, k, rng):
        x0, y0, x1, y1 = geom.bounds
        pts, new = list(prev), []
        for _ in range(k):
            c = np.empty((0, 2))
            while len(c) < m:
                xy = rng.uniform((x0, y0), (x1, y1), (2 * m, 2))
                c = np.r_[c, xy[shapely.contains_xy(geom, xy[:, 0], xy[:, 1])]]
            c = c[:m]
            if pts: c = c[[((stretch(c)[:, None] - stretch(np.array(pts))[None])**2).sum(-1).min(1).argmax()]]
            new.append(c[0]); pts.append(c[0])
        return np.array(new)
    return place

`Campaign.place` calls the campaign's `placement` for every unit with samples to draw, passing the coordinates of the unit's earlier samples, and returns the locations in the form `Sampler.sample` gives:

In [ ]:
@patch
def place(self:Campaign,
    n:np.ndarray, # Number of samples for each unit, in `gdf_units` order
) -> gpd.GeoDataFrame: # One `MultiPoint` per unit with `n` above 0, indexed by `loc_id`
    "Sample locations from `placement`, given each unit's earlier samples."
    prev = self.samples
    xy = {} if prev.empty else {i: np.c_[shapely.get_x(g.values), shapely.get_y(g.values)] for i, g in prev.geometry.groupby(level=0)}
    geoms = {loc_id: shapely.MultiPoint(self.placement(geom, xy.get(loc_id, np.empty((0, 2))), int(k), self.rng))
             for (loc_id, geom), k in zip(self.gdf_units.geometry.items(), n) if k > 0}
    return gpd.GeoDataFrame(geometry=gpd.GeoSeries(geoms, crs=self.gdf_units.crs)).rename_axis('loc_id')

Three rounds of 3 samples per unit, placed uniformly and with `spread`. The mean distance from each sample to its nearest neighbour in the same unit is larger with `spread`, and every unit gets its 9 samples:

In [ ]:
def nearest_dist(samples):
    "Mean distance from each sample to the nearest other sample in its unit."
    def unit(g):
        xy = np.c_[shapely.get_x(g.values), shapely.get_y(g.values)]
        d = np.sqrt(((xy[:, None] - xy[None])**2).sum(-1)); np.fill_diagonal(d, np.inf)
        return d.min(1).mean()
    return samples.geometry.groupby(level=0).apply(unit).mean()

dists = {}
for name, placement in {'uniform': None, 'spread': spread()}.items():
    c = Campaign(fname_raster, gdf_grid, placement=placement); c.reset(0)
    for _ in range(3): c.step(np.full(len(gdf_grid), 3))
    test_eq(c.samples.groupby(level=0).size().unique(), [9])
    dists[name] = nearest_dist(c.samples)
assert dists['spread'] > 1.3 * dists['uniform']
dists

A strategy is a function that takes the samples so far, the units, the round's budget and the campaign's random generator, and returns the number of samples for each unit. `mcdm` ranks the units by MCDM on the criteria that `cbs` computes and splits the budget with `rank_to_sample`. While there are no samples, it ranks from the prior alone. The prior is the ground truth itself until #11 adds a degraded prior. The defaults give the strategy of [Run a campaign over several years](how-to/01_multi_year_campaign.ipynb):


In [ ]:
def mcdm(
    fname_prior:str, # Raster used as the prior
    cbs:list=None, # Callbacks computing the criteria once there are samples. `None` means Max, Min, Std, Count, Moran's I and Prior
    is_benefit:list=[True, True, True, False, False, True], # Benefit (`True`) or cost (`False`) flag of each criterion
    weights:list=[0.2, 0.1, 0.1, 0.2, 0.2, 0.2], # Weight of each criterion
    policy:str='quantiles', # Allocation policy of `rank_to_sample`
    min:int=1, # Minimum number of samples per unit
):
    "Strategy ranking units by MCDM, from the prior alone while there are no samples."
    cbs = cbs or [MaxCB(), MinCB(), StdCB(), CountCB(), MoranICB(k=5), PriorCB(fname_prior)]
    def strategy(samples, gdf_units, budget, rng):
        if samples.empty: cbs_, isb, w = [PriorCB(fname_prior)], [True], [1]
        else: cbs_, isb, w = cbs, is_benefit, weights
        state = State(samples, gdf_units, cbs=cbs_)()
        ranks = Optimizer(state=state).get_rank(is_benefit_x=isb, w_vector=w, n_method='LINEAR1', s_method='CP')
        return rank_to_sample(ranks['rank'].sort_index().values, budget=budget, min=min, policy=policy)
    return strategy


`run` plays a strategy for one budget per round and returns the per-unit tables of every round, indexed by `round` and `loc_id`:

In [ ]:
def run(
    campaign:Campaign, # Campaign to play
    strategy:callable, # Takes samples so far, units, budget and random generator; returns samples per unit
    budgets:list, # Budget of each round
    seed:int=None, # Seed passed to `Campaign.reset`
) -> pd.DataFrame: # `unit_stats` after each round, indexed by `round` and `loc_id`
    "Play `strategy` on `campaign` for `len(budgets)` rounds."
    campaign.reset(seed)
    tables = {r: campaign.step(strategy(campaign.samples, campaign.gdf_units, b, campaign.rng)) for r, b in enumerate(budgets, 1)}
    return pd.concat(tables, names=['round'])

Six rounds of 150 samples with the how-to strategy. The same seed gives the same campaign:

In [ ]:
campaign = Campaign(fname_raster, gdf_grid)
res = run(campaign, mcdm(fname_raster), budgets=[150] * 6, seed=0)
test_eq(res.loc[6].n.sum(), len(campaign.samples))
test_eq(campaign.samples['round'].max(), 6)
test_eq(run(campaign, mcdm(fname_raster), budgets=[150] * 6, seed=0), res)

`summarise` averages the errors over units after each round:

In [ ]:
def summarise(res): 
    "Mean over units of the errors in each round of `res`."
    return res.assign(abs_err_mean=(res.meas_mean - res.true_mean).abs(), err_max=res.meas_max - res.true_max
        ).groupby('round').agg(samples=('n', 'sum'), unsampled=('n', lambda n: (n == 0).sum()), mean_abs_err_mean=('abs_err_mean', 'mean'),
                               mean_err_max=('err_max', 'mean'), mean_wasserstein=('wasserstein', 'mean'))

summarise(res).round(4)

## Baseline strategies

The baselines place the MCDM strategies between allocations that need no ranking. `allocate` gives each unit `min` samples, then splits the rest of the budget in proportion to a weight per unit. It rounds each share down and gives the samples left over to the units with the largest remainders. A random amount far below one sample breaks ties between equal remainders:

In [ ]:
def allocate(
    weights:np.ndarray, # Weight of each unit
    budget:int, # Total number of samples
    min:int=1, # Minimum number of samples per unit
    rng:np.random.Generator=None, # Generator breaking ties between equal remainders
) -> np.ndarray: # Number of samples for each unit, summing to `budget`
    "Split `budget` across units in proportion to `weights`, after `min` samples each."
    weights = np.asarray(weights, dtype=float)
    if weights.sum() == 0: weights = np.ones_like(weights)
    n = np.full(len(weights), min)
    if budget < n.sum(): raise ValueError(f"`budget` {budget} is below {n.sum()}, the minimum over all units")
    share = (budget - n.sum()) * weights / weights.sum()
    n += np.floor(share).astype(int)
    rem = share - np.floor(share) + np.random.default_rng(rng).random(len(share)) * 1e-9
    n[np.argsort(-rem)[:budget - n.sum()]] += 1
    return n

In [ ]:
n = allocate([1, 1, 1, 3], 10, rng=0)
test_eq(n.sum(), 10)
n

`equal` gives every unit the same number of samples, up to one sample of rounding. Every other strategy should do at least as well:

In [ ]:
def equal(
    min:int=1, # Minimum number of samples per unit in each round
):
    "Strategy giving every unit the same share of the budget."
    def strategy(samples, gdf_units, budget, rng): return allocate(np.ones(len(gdf_units)), budget, min=min, rng=rng)
    return strategy


The textbook allocation for estimating each unit's mean gives a unit samples in proportion to its standard deviation. On units of equal size this is Neyman allocation. `neyman` estimates the standard deviation from the samples so far. A unit with fewer than 2 samples gets the largest estimate, and the first round, with no samples at all, is equal allocation. `oracle` uses the true standard deviation of each unit from the ground truth. No real campaign can do this, so it is an upper reference. Both split each round's budget on its own, without accounting for the samples of earlier rounds:

In [ ]:
def neyman(
    min:int=1, # Minimum number of samples per unit in each round
):
    "Strategy allocating in proportion to each unit's standard deviation, estimated from the samples so far."
    def strategy(samples, gdf_units, budget, rng):
        if samples.empty: return allocate(np.ones(len(gdf_units)), budget, min=min, rng=rng)
        std = samples.value.astype(float).groupby(level=0).std().reindex(gdf_units.index)
        return allocate(std.fillna(std.max()).values, budget, min=min, rng=rng)
    return strategy

def oracle(
    fname_raster:str, # Ground-truth raster
    min:int=1, # Minimum number of samples per unit in each round
):
    "Strategy allocating in proportion to each unit's true standard deviation."
    std = {}
    def strategy(samples, gdf_units, budget, rng):
        for loc_id, geom in gdf_units.geometry.items():
            if loc_id not in std: std[loc_id] = zonal_values(fname_raster, geom).std()
        return allocate(gdf_units.index.map(std).values, budget, min=min, rng=rng)
    return strategy


## Comparing strategies

`compare` runs each strategy with each seed and keeps the `summarise` table of every run. Every strategy gives each unit at least one sample per round, as the how-to strategy does:


In [ ]:
def compare(
    campaign:Campaign, # Campaign to play
    strategies:dict, # Strategies by name
    budgets:list, # Budget of each round
    seeds:list, # One run per seed
    summary:callable=summarise, # Summarises one run's per-unit tables by round
) -> pd.DataFrame: # `summary` of every run, indexed by `strategy`, `seed` and `round`
    "Run every strategy in `strategies` once per seed."
    return pd.concat({(name, seed): summary(run(campaign, strategy, budgets, seed))
                      for name, strategy in strategies.items() for seed in seeds}, names=['strategy', 'seed'])

## Minimum number of samples

A minimum per round fixes part of every round's budget: with 100 units, `min=1` fixes 100 of 150 samples in each round, and 600 of 900 over six rounds. The reasons for a minimum concern the samples a unit has in total: each unit needs some measurement for its decision, `StdCB` and `MoranICB` need several samples, and a minimum protects against a wrong prior. `cumulative_min` tops up every unit with fewer than `k` samples so far, then lets the wrapped strategy target the rest of the budget. When the top-up needs the whole budget, the budget goes to the units in proportion to what they lack. The wrapped strategy does not see the round's top-up when it allocates:

In [ ]:
def cumulative_min(
    strategy:callable, # Strategy targeting the budget left after the top-up
    k:int, # Minimum number of samples per unit over the whole campaign
):
    "Strategy topping up every unit to `k` samples in total before `strategy` targets the rest of the budget."
    def wrapped(samples, gdf_units, budget, rng):
        have = samples.groupby(level=0).size().reindex(gdf_units.index, fill_value=0).values
        top = np.maximum(k - have, 0)
        if top.sum() >= budget: return allocate(top, budget, min=0, rng=rng)
        return top + strategy(samples, gdf_units, budget - top.sum(), rng)
    return wrapped

## Decision score

The DSS4NAFA restrictions dashboard suggests an action for each unit from three settings: a permissible level, a minimum number of samples and an accepted proportion over the permissible level. `decide` applies that rule to a unit's values. A unit with no values is `undefined`, and one with fewer than `min_n` is `sample`. Otherwise the unit is restricted when the share of its values above `level` exceeds `accepted`. The dashboard's "Partially restrict" is left out:

In [ ]:
def decide(
    values:np.ndarray, # Values measured in one unit
    level:float, # Permissible level
    min_n:int=1, # Minimum number of values for a decision
    accepted:float=0.1, # Accepted proportion of values above `level`
) -> str: # `restrict`, `no restrict`, `sample` or `undefined`
    "Action the DSS4NAFA dashboard suggests for one unit."
    if len(values) == 0: return 'undefined'
    if len(values) < min_n: return 'sample'
    return 'restrict' if (values > level).mean() > accepted else 'no restrict'

In [ ]:
test_eq(decide(np.array([0.5, 0.9, 0.7]), level=0.8, accepted=0.1), 'restrict')
test_eq(decide(np.array([0.5, 0.9, 0.7]), level=0.8, accepted=0.5), 'no restrict')
test_eq(decide(np.array([0.9]), level=0.8, min_n=3), 'sample')

`decisions` applies the rule twice for each unit: to the samples, and to the true values, which give the true action. It is a `score` for `Campaign`:

In [ ]:
def decisions(
    samples:gpd.GeoDataFrame, # Measurements indexed by `loc_id`, with a `value` column
    truth:pd.Series, # Raster cell values of each unit, from `unit_values`
    level:float=0.8, # Permissible level
    min_n:int=1, # Minimum number of samples for a decision
    accepted:float=0.1, # Accepted proportion of values above `level`
) -> pd.DataFrame: # One row per unit
    "True and suggested action of each unit, with the true and measured shares above `level`."
    rows = {}
    for loc_id, true in truth.items():
        meas = samples.value[samples.index == loc_id].values.astype(float)
        rows[loc_id] = dict(n=len(meas), true_share=(true > level).mean(), share=(meas > level).mean() if len(meas) else np.nan,
                            true_action=decide(true, level, accepted=accepted), action=decide(meas, level, min_n, accepted))
    return pd.DataFrame.from_dict(rows, orient='index').rename_axis('loc_id')

`summarise_decisions` counts, after each round, the units restricted although the truth says no (`needless`), the units not restricted although the truth says restrict (`missed`), and the units left without a decision (`unresolved`):

In [ ]:
def summarise_decisions(res):
    "Counts of needless, missed and unresolved decisions in each round of `res`."
    return res.assign(needless=(res.action == 'restrict') & (res.true_action == 'no restrict'),
                      missed=(res.action == 'no restrict') & (res.true_action == 'restrict'),
                      unresolved=res.action.isin(['sample', 'undefined'])
                     ).groupby('round')[['needless', 'missed', 'unresolved']].sum().assign(wrong=lambda d: d.needless + d.missed)

## Decision-aware strategies

`bayes` targets the units whose decision is most likely to be wrong. For each unit, with `k` of its `n` samples above `level`, the share of the unit above `level` gets a beta posterior. Without a prior raster, it starts from Beta(0.5, 0.5), which carries no information, and the posterior is Beta(0.5 + k, 0.5 + n - k). With `fname_prior`, a unit whose prior raster has a share `s0` of its cells above `level` starts from Beta(0.5 + m s0, 0.5 + m (1 - s0)), where `m` is `strength`, the number of samples the prior is worth. The probability that the dashboard's decision is wrong is the smaller of the posterior probability that the share exceeds `accepted` and its complement. `bayes` splits the budget in proportion to it. Without a prior and before any sample, every unit has the same probability, and the first round is equal allocation:

In [ ]:
def bayes(
    level:float=0.8, # Permissible level
    accepted:float=0.1, # Accepted proportion of values above `level`
    min:int=1, # Minimum number of samples per unit in each round
    fname_prior:str=None, # Prior raster; `None` for no prior information
    strength:float=4, # Number of samples the prior is worth
):
    "Strategy allocating in proportion to the posterior probability that each unit's decision is wrong."
    s0 = None
    def strategy(samples, gdf_units, budget, rng):
        nonlocal s0
        above = samples.value.astype(float) > level
        n = above.groupby(level=0).size().reindex(gdf_units.index, fill_value=0)
        k = above.groupby(level=0).sum().reindex(gdf_units.index, fill_value=0)
        if fname_prior and s0 is None: s0 = unit_values(fname_prior, gdf_units).map(lambda v: (v > level).mean()).astype(float)
        a0, b0 = (0.5 + strength * s0, 0.5 + strength * (1 - s0)) if fname_prior else (0.5, 0.5)
        p = beta.sf(accepted, a0 + k, b0 + n - k)
        return allocate(np.minimum(p, 1 - p), budget, min=min, rng=rng)
    return strategy

After the six rounds of the how-to campaign above, `bayes` gives the largest shares of a 150-sample round to the units whose measured share above 0.8 is near 10%. The rest of the budget spreads widely: with a few samples, the posterior on a share stays wide. A unit with 6 samples, none above the level, still has a 25% probability of a share above 10%:


In [ ]:
nb = bayes(min=0)(campaign.samples, gdf_grid, 150, np.random.default_rng(0))
test_eq(nb.sum(), 150)
top = decisions(campaign.samples, campaign.truth).assign(next=nb).nlargest(5, 'next')
top[['n', 'share', 'true_share', 'next']]

In [ ]:
test_close(beta.sf(0.1, 0.5, 6.5), 0.25, eps=0.01)

`decision_oracle` knows each unit's true share above `level`. With `n` samples, the number above `level` follows a binomial distribution, which gives the probability that the dashboard's rule reaches the wrong decision. A unit without samples counts as wrong. It splits the budget in proportion to that probability:

In [ ]:
def decision_oracle(
    fname_raster:str, # Ground-truth raster
    level:float=0.8, # Permissible level
    accepted:float=0.1, # Accepted proportion of values above `level`
    min:int=1, # Minimum number of samples per unit in each round
):
    "Strategy allocating in proportion to the true probability that each unit's decision is wrong."
    q = None
    def strategy(samples, gdf_units, budget, rng):
        nonlocal q
        if q is None: q = unit_values(fname_raster, gdf_units).map(lambda v: (v > level).mean()).astype(float).values
        n = samples.groupby(level=0).size().reindex(gdf_units.index, fill_value=0).values
        p_restrict = binom.sf(np.floor(accepted * n), n, q)
        wrong = np.where(q > accepted, 1 - p_restrict, p_restrict)
        wrong[n == 0] = 1
        return allocate(wrong, budget, min=min, rng=rng)
    return strategy

## Degraded priors

So far the prior raster has been the ground truth itself, which no real campaign has. `degrade` writes a prior that resembles an airborne survey: it averages the ground truth over blocks of `factor` by `factor` cells, multiplies each block by lognormal noise with median 1 and by `bias`, and writes the result on the ground truth's grid. `PriorCB`, `mcdm` and `bayes` take the prior raster as any other raster:

In [ ]:
def degrade(
    fname_truth:str, # Ground-truth raster
    dst:str, # Path of the prior raster to write
    factor:int=10, # Side of the blocks averaged together, in cells
    noise:float=0.3, # Standard deviation of the log of the multiplicative noise, one draw per block
    bias:float=1.0, # Factor applied to every value
    seed:int=None, # Seed for the noise
) -> str: # `dst`
    "Write a prior for `fname_truth`: block averages times lognormal noise and `bias`, on the ground truth's grid."
    with rasterio.open(fname_truth) as src:
        prof, h, w = src.profile, src.height, src.width
        coarse = src.read(1, out_shape=(math.ceil(h / factor), math.ceil(w / factor)), resampling=Resampling.average).astype(float)
    coarse *= bias * np.exp(np.random.default_rng(seed).normal(0, noise, coarse.shape))
    prior = np.repeat(np.repeat(coarse, factor, axis=0), factor, axis=1)[:h, :w]
    with rasterio.open(dst, 'w', **(prof | dict(dtype='float32'))) as out: out.write(prior.astype('float32'), 1)
    return dst

A prior of the simulated raster with blocks of 10 cells and 30% noise. Unit means of the prior follow those of the ground truth, with errors of 0.07 on average and up to 0.28, on values between 0 and 1:


In [ ]:
fname_prior = degrade(fname_raster, os.path.join(tempfile.mkdtemp(), 'prior.tif'), factor=10, noise=0.3, seed=0)
means = pd.DataFrame({'truth': unit_values(fname_raster, gdf_grid).map(np.mean), 'prior': unit_values(fname_prior, gdf_grid).map(np.mean)})
assert means.corr().iloc[0, 1] > 0.9
(means.prior - means.truth).abs().describe()[['mean', 'max']].round(3)

With that prior, `bayes` no longer starts with equal allocation: the units whose prior share above the level is near 10% get the most samples in round 1. A prior worth 4 samples is weak, though. A unit whose prior has no cell above the level still has a 34% probability of a share above 10%, and the first round still reaches 61 of the 100 units. A prior worth 20 samples would lower that probability to 4%:


In [ ]:
n1 = bayes(fname_prior=fname_prior, min=0)(campaign.samples.iloc[:0], gdf_grid, 150, np.random.default_rng(0))
prior_share = unit_values(fname_prior, gdf_grid).map(lambda v: (v > 0.8).mean())
pd.DataFrame({'prior_share': prior_share, 'next': n1}).nlargest(5, 'next').round(3)

In [ ]:
test_close(beta.sf(0.1, 0.5, 4.5), 0.34, eps=0.01)
test_close(beta.sf(0.1, 0.5, 20.5), 0.04, eps=0.01)

## Predicting the dashboard's decision

`bayes` allocates on the posterior probability that a unit's share exceeds the accepted proportion. The dashboard decides from the samples alone, so that probability is not the probability of a wrong decision. With a confident prior, a unit can be almost surely above the level and still be decided by one sample. `expected_wrong` computes the probability that matters: that the dashboard's rule is wrong for a unit after `m` more samples. For each share `q` on a grid, the `m` new samples bring a binomial number of exceedances, which sets the decision. The decision is wrong when it differs from the true one, `q` above `accepted`. The result averages over the posterior on `q`. A unit with no samples at all is undefined and counts as wrong:

In [ ]:
def expected_wrong(
    k:int, # Samples above the level so far
    n:int, # Samples so far
    m:int, # Samples to add
    a0:float=0.5, # First parameter of the prior on the share
    b0:float=0.5, # Second parameter of the prior on the share
    accepted:float=0.1, # Accepted proportion of values above the level
    grid:np.ndarray=np.linspace(0.0025, 0.9975, 200), # Shares to integrate over
) -> float: # Probability of a wrong decision after `m` more samples
    "Expected probability that the dashboard's rule is wrong for a unit after `m` more samples."
    if n + m == 0: return 1.0
    w = beta.pdf(grid, a0 + k, b0 + n - k)
    w /= w.sum()
    needed = np.floor(accepted * (n + m) - k) + 1
    p_restrict = binom.sf(needed - 1, m, grid)
    return float(w @ np.where(grid > accepted, 1 - p_restrict, p_restrict))

`expected_wrong` sees the sawtooth. A unit with 1 of 9 samples above the level is restricted. One more sample raises the probability of a wrong decision, from 0.39 to 0.50, because the 10th sample raises the count needed to 2. It takes three more samples to bring it below where it started:


In [ ]:
ew = [expected_wrong(1, 9, m) for m in range(4)]
assert ew[1] > ew[2] > ew[0] > ew[3]
np.round(ew, 3)

`voi` tops each unit up to `min_total` samples in total, then gives the rest of the budget one sample at a time. Each sample goes to the unit with the largest gain: the largest average drop of `expected_wrong` per sample over the next 1 to `horizon` samples. Looking several samples ahead matters near a jump of the sawtooth, where the next sample alone raises the error although a few more lower it. It also matters for a unit with an early exceedance: with 1 sample above the level out of 3, the measured share drops to 10% only after 7 more samples below it. With a horizon of 5, `voi` saw no gain there and left such units restricted needlessly. The default of 10 comes from the plume experiments in `_scoring_experiments.ipynb`. When no unit gains anything, the rest of the budget goes in proportion to each unit's current probability of a wrong decision. A random amount far below any real gain breaks ties. With `fname_prior`, units start from the same prior as in `bayes`:


In [ ]:
def voi(
    level:float=0.8, # Permissible level
    accepted:float=0.1, # Accepted proportion of values above `level`
    min_total:int=1, # Minimum number of samples per unit over the whole campaign
    fname_prior:str=None, # Prior raster; `None` for no prior information
    strength:float=4, # Number of samples the prior is worth
    horizon:int=10, # Number of samples to look ahead when comparing units
):
    "Strategy giving each sample to the unit where it most lowers the expected probability of a wrong decision."
    s0 = None
    def strategy(samples, gdf_units, budget, rng):
        nonlocal s0
        above = samples.value.astype(float) > level
        n = above.groupby(level=0).size().reindex(gdf_units.index, fill_value=0).values
        k = above.groupby(level=0).sum().reindex(gdf_units.index, fill_value=0).values
        if fname_prior and s0 is None: s0 = unit_values(fname_prior, gdf_units).map(lambda v: (v > level).mean()).astype(float).values
        a0, b0 = (0.5 + strength * s0, 0.5 + strength * (1 - s0)) if fname_prior else (np.full(len(n), 0.5), np.full(len(n), 0.5))
        m = np.maximum(min_total - n, 0)
        if m.sum() >= budget: return allocate(m, budget, min=0, rng=rng)
        ew = lambda u, mm: expected_wrong(k[u], n[u], mm, a0[u], b0[u], accepted)
        cur = np.array([ew(u, m[u]) for u in range(len(n))])
        gain = lambda u: max((cur[u] - ew(u, m[u] + j)) / j for j in range(1, horizon + 1))
        g = np.array([gain(u) for u in range(len(n))])
        for left in range(budget - m.sum(), 0, -1):
            if g.max() <= 0: return m + allocate(cur, left, min=0, rng=rng)
            u = np.argmax(g + rng.random(len(n)) * 1e-12)
            m[u] += 1
            cur[u] = ew(u, m[u])
            g[u] = gain(u)
        return m
    return strategy

After the six rounds of the how-to campaign above, `voi` gives a 150-sample round to the units whose decision can still change: units with 6 samples and none above the level, which still have a 25% probability of a share above 10%, and units whose measured share is near 10%:


In [ ]:
nv = voi()(campaign.samples, gdf_grid, 150, np.random.default_rng(0))
test_eq(nv.sum(), 150)
decisions(campaign.samples, campaign.truth).assign(next=nv).nlargest(5, 'next')[['n', 'share', 'true_share', 'next']].round(3)